In [0]:
dbutils.widgets.text("env", "brz")
ENV = dbutils.widgets.get("env")
CATALOG = "elixir"
data_path = f'{CATALOG}.{ENV}.topdesk_incidents_raw'


In [0]:
# we read in the data
df_bronze = spark.table(data_path)

In [0]:
display(df_bronze)

In [0]:
# we parse json into columns
from pyspark.sql.functions import parse_json, from_json, schema_of_json, col
# Infer schema from one record
sample_json = df_bronze.select("raw_json").limit(1).collect()[0][0]
json_schema = schema_of_json(sample_json)
# Parse JSON string into columns
parsed_df = df_bronze.withColumn("data", from_json(col("raw_json"), json_schema)).select("data.*", "ingestion_time")

display(parsed_df)

In [0]:
clean_df = parsed_df.select(
    col("id").alias("incident_id"),
    col("category.name").alias("category"),
    col("callerBranch.id").alias("caller_branch_id"),
    col("callerBranch.name").alias("caller_branch_name"),
    col("callerBranch.timeZone").alias("caller_branch_timezone"),
    col("status"),
    col("sla.id").alias("sla_id"),
    col("sla.responseTargetDate").alias("sla_response_date"),
    col("sla.targetDate").alias("sla_target"),
       col("processingStatus.id").alias("processing_id"),
    col("processingStatus.name").alias("processing_name"),
    col("creationDate"),
    col("modificationDate")
    
    )


In [0]:
from pyspark.sql.functions import to_timestamp

clean_df = clean_df.withColumn(
    "modification_ts", to_timestamp("modificationDate")
).withColumn(
    "creation_ts", to_timestamp("creationDate")
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("incident_id").orderBy(desc("modification_ts"))

dedup_df = clean_df.withColumn( "rn", row_number().over(window)).filter("rn = 1").drop("rn")

In [0]:
# count the size difference between dedup and clean

clean_df.count(), dedup_df.count()

In [0]:
# no null ids
dedup_df.filter("incident_id IS NULL").count()

# duplicates
dedup_df.groupBy("incident_id").count().filter("count > 1")

In [0]:
from pyspark.sql.functions import col, sum as _sum, when

null_counts = dedup_df.select([_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in dedup_df.columns])
null_counts.show()


In [0]:
display(dedup_df.select("category").distinct())

In [0]:
# show unqiue rows in the processing_name 
display(dedup_df.select("processing_name").distinct())

In [0]:
closed_tickets = dedup_df.where(col("processing_name") == "Closed")
closed_tickets.count()


In [0]:
from pyspark.sql.functions import col, lower
filtered_ocg_oca = closed_tickets.filter(
    ((lower(col("category")) .startswith("ocg")) |
     (lower(col("category")) .startswith("oca")))
    & (lower(col("category")).contains("unifield sup"))
)
display(filtered_ocg_oca)